<a href="https://colab.research.google.com/github/dkang1630/Conductor_Image_Classification/blob/main/U_Net_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import random
from google.colab import drive
import numpy as np
import cv2
from glob import glob
import tensorflow as tf
import shutil
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPool2D, Conv2DTranspose, Concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#checking if images match up
base_path = "/content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net"
original_base_path = os.path.join(base_path, "Original")
SAM_base_path = os.path.join(base_path, "SAM")

# Get the list of image files from both folders
original_images = [f for f in os.listdir(original_base_path) if f.endswith('.jpg')]
SAM_images = [f for f in os.listdir(SAM_base_path) if f.endswith('.jpg')]

# Remove "Copy of " from SAM image names to align with original image names
SAM_images_cleaned = [f.replace("Copy of ", "") for f in SAM_images]

# Find mismatches
original_set = set(original_images)
SAM_set = set(SAM_images_cleaned)

missing_in_SAM = original_set - SAM_set
extra_in_SAM = SAM_set - original_set

# Print results
if not missing_in_SAM and not extra_in_SAM:
    print("All images match between the Original and SAM folders!")
else:
    if missing_in_SAM:
        print(f"Images missing in SAM folder: {missing_in_SAM}")
    if extra_in_SAM:
        print(f"Extra images in SAM folder: {extra_in_SAM}")

All images match between the Original and SAM folders!


In [ ]:
#Set random seed
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Numpy random array:", np.random.rand(3))  # Generate a random array with 3 elements
print("Python random number:", random.random())  # Generate a random float

Numpy random array: [0.37454012 0.95071431 0.73199394]
Python random number: 0.6394267984578837


In [ ]:
# Define base paths
base_path = "/content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net"
original_base_path = os.path.join(base_path, "Original")
binary_mask_base_path = os.path.join(base_path, "Binary_Masked")

original_train_base_path = os.path.join(original_base_path, "train")
original_validation_base_path = os.path.join(original_base_path, "validation")

binary_mask_train_base_path = os.path.join(binary_mask_base_path, "train")
binary_mask_validation_base_path = os.path.join(binary_mask_base_path, "validation")

files_dir = os.path.join(base_path, "Files")
model_file = os.path.join(files_dir, "Unet_model.keras")
csv_file = os.path.join(files_dir, "Unet_log.csv")


In [ ]:
#Dont run this unless you need to reshuffle
# Get lists of files
original_files = sorted([f for f in os.listdir(original_base_path) if f.endswith('.jpg')])
binary_mask_files = sorted([f for f in os.listdir(binary_mask_base_path) if f.endswith('.jpg')])

#verify matching pairs
assert len(original_files) == len(binary_mask_files), "Mismath in length between original and mask files"

#Create paired list and shuffle
paired_files = list(zip(original_files, binary_mask_files))
random.seed(42)
random.shuffle(paired_files)

#split 80-20
train_files, val_files = train_test_split(paired_files, test_size=0.2, random_state=42)

def copy_files(file_pairs, original_dest, mask_dest):
    for original_file, mask_file in file_pairs:
        #copy original images
        shutil.copy2(
            os.path.join(original_base_path, original_file),
            os.path.join(original_dest, original_file)
        )
        #copy corresponding mask
        shutil.copy2(
            os.path.join(binary_mask_base_path, mask_file),
            os.path.join(mask_dest, mask_file)
        )

#Training files
copy_files(train_files,
           os.path.join(original_base_path, "train"),
           os.path.join(binary_mask_base_path, "train"))

#validation
copy_files(val_files,
           os.path.join(original_base_path, "validation"),
           os.path.join(binary_mask_base_path, "validation"))

print("Splitting complete")
print(f"Training images: {len(train_files)}")
print(f"Validation images: {len(val_files)}")


Splitting complete
Training images: 220
Validation images: 56


In [ ]:
# hyperparameter
batch_size = 8
lr = 1e-4
epochs = 100
height = 128
width = 128

In [ ]:
#Conv Block
def conv_block(inputs, num_filters):
    x = Conv2D(num_filters, 3, padding="same")(inputs)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    x = Conv2D(num_filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)

    return x

#Encoder Block
def encoder_block(inputs, num_filters):
    x = conv_block(inputs, num_filters)
    p = MaxPool2D((2, 2))(x)
    return x, p

#Decoder Block
def decoder_block(inputs, skip_features, num_filters):
    x = Conv2DTranspose(num_filters, (2, 2), strides=2, padding="same")(inputs)
    x = Concatenate() ([x, skip_features])
    x = conv_block(x, num_filters)
    return x

In [ ]:
def build_unet_model(input_shape):
    inputs = Input(input_shape)

    #Encoder
    s1, p1 = encoder_block(inputs, 64)
    s2, p2 = encoder_block(p1, 128)
    s3, p3 = encoder_block(p2, 256)
    s4, p4 = encoder_block(p3, 512)

    #Bridge
    b1 = conv_block(p4, 1024)

    #Decoder
    d1 = decoder_block(b1, s4, 512)
    d2 = decoder_block(d1, s3, 256)
    d3 = decoder_block(d2, s2, 128)
    d4 = decoder_block(d3, s1, 64)

    #Output
    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4)

    model = Model(inputs, outputs, name="U-Net")
    return model

In [ ]:
#Loading training and validation datset
def load_data(path):
    train_images = sorted(glob(os.path.join(path, "Original", "train", "*.jpg")))
    train_masks = sorted(glob(os.path.join(path, "Binary_Masked", "train", "*.jpg")))

    valid_images = sorted(glob(os.path.join(path, "Original", "validation", "*.jpg")))
    valid_masks = sorted(glob(os.path.join(path, "Binary_Masked", "validation", "*.jpg")))
    return (train_images, train_masks), (valid_images, valid_masks)

In [ ]:
!ls /content/drive/My\ Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Original/


img_100.jpg  img_144.jpg  img_190.jpg  img_234.jpg  img_286.jpg  img_375.jpg  img_67.jpg
img_101.jpg  img_145.jpg  img_193.jpg  img_235.jpg  img_287.jpg  img_377.jpg  img_68.jpg
img_102.jpg  img_146.jpg  img_194.jpg  img_236.jpg  img_288.jpg  img_378.jpg  img_69.jpg
img_103.jpg  img_147.jpg  img_195.jpg  img_238.jpg  img_289.jpg  img_37.jpg   img_70.jpg
img_104.jpg  img_148.jpg  img_196.jpg  img_241.jpg  img_28.jpg	 img_381.jpg  img_71.jpg
img_105.jpg  img_149.jpg  img_197.jpg  img_244.jpg  img_290.jpg  img_384.jpg  img_72.jpg
img_106.jpg  img_150.jpg  img_198.jpg  img_247.jpg  img_291.jpg  img_387.jpg  img_73.jpg
img_107.jpg  img_151.jpg  img_199.jpg  img_250.jpg  img_292.jpg  img_396.jpg  img_74.jpg
img_108.jpg  img_152.jpg  img_19.jpg   img_251.jpg  img_293.jpg  img_39.jpg   img_75.jpg
img_109.jpg  img_153.jpg  img_1.jpg    img_252.jpg  img_294.jpg  img_405.jpg  img_76.jpg
img_10.jpg   img_154.jpg  img_200.jpg  img_253.jpg  img_295.jpg  img_40.jpg   img_77.jpg
img_110.jpg  img_155.j

In [ ]:
def read_image(path):
    path = path.decode()
    x = cv2.imread(path, cv2.IMREAD_COLOR)
    x = x/255.0
    return x

def read_mask(path):
    path = path.decode()
    x = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    x = x/255.0
    x = np.expand_dims(x, axis=-1)
    return x

In [ ]:
# Data Augmentation: Applied only during training
def data_augmentation(image, mask):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.3)
    image = tf.image.random_hue(image, max_delta=0.1)
    image = tf.image.random_saturation(image, lower=0.7, upper=1.3)

    return image, mask

In [ ]:
# Function to resize images to the correct shape (128x128)
def resize_image(image, mask, target_size=(128, 128)):
    image = tf.image.resize(image, target_size)
    mask = tf.image.resize(mask, target_size)
    return image, mask

#data pipeline
def tf_parse(x, y):
    def _parse(x, y):
        x = read_image(x)
        y = read_mask(y)
        x, y = resize_image(x, y)  # Ensure both image and mask are resized
        x = tf.cast(x, tf.float64)  # Cast to float64
        y = tf.cast(y, tf.float64)  # Cast to float64
        return x, y

    x, y = tf.numpy_function(_parse, [x, y], [tf.float64, tf.float64])
    x.set_shape([height, width, 3])
    y.set_shape([height, width, 1])
    return x, y

def tf_dataset(images, masks, batch=8, augment=False):
    dataset = tf.data.Dataset.from_tensor_slices((images, masks))
    dataset = dataset.map(tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        dataset = dataset.map(lambda x, y: data_augmentation(x, y))  # Apply augmentation

    dataset = dataset.batch(batch)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
#check number of training and validation dataset
(train_images, train_masks), (valid_images, valid_masks) = load_data(base_path)
print(f"Train: {len(train_images)} - {len(train_masks)}")
print(f"Valid: {len(valid_images)} - {len(valid_masks)}")

Train: 220 - 220
Valid: 56 - 56


In [ ]:
train_dataset = tf_dataset(train_images, train_masks, batch=batch_size, augment=True)
valid_dataset = tf_dataset(valid_images, valid_masks, batch=batch_size, augment=False)

In [ ]:
for x, y in valid_dataset:  # Take a single batch
    print(x.shape, y.shape)  # This should print (batch_size, 128, 128, 3) for images and (batch_size, 128, 128, 1) for masks

(8, 128, 128, 3) (8, 128, 128, 1)
(8, 128, 128, 3) (8, 128, 128, 1)
(8, 128, 128, 3) (8, 128, 128, 1)
(8, 128, 128, 3) (8, 128, 128, 1)
(8, 128, 128, 3) (8, 128, 128, 1)
(8, 128, 128, 3) (8, 128, 128, 1)
(8, 128, 128, 3) (8, 128, 128, 1)


In [ ]:
input_shape = (height, width, 3)
model = build_unet_model(input_shape)
model.summary()

Model: "U-Net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 128, 128, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d (Conv2D)           │ (None, 128, 128, 64)   │          1,792 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 128, 128, 64)   │            256 │ conv2d[0][0]           │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation (Activation)   │ (None, 128, 128, 64)   │              0 │ batch_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_1 (Conv2D)         │ (None, 128, 128, 64)   │         36,928 │ activation[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 128, 128, 64)   │            256 │ conv2d_1[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_1 (Activation) │ (None, 128, 128, 64)   │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d             │ (None, 64, 64, 64)     │              0 │ activation_1[0][0]     │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_2 (Conv2D)         │ (None, 64, 64, 128)    │         73,856 │ max_pooling2d[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_2     │ (None, 64, 64, 128)    │            512 │ conv2d_2[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_2 (Activation) │ (None, 64, 64, 128)    │              0 │ batch_normalization_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_3 (Conv2D)         │ (None, 64, 64, 128)    │        147,584 │ activation_2[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_3     │ (None, 64, 64, 128)    │            512 │ conv2d_3[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_3 (Activation) │ (None, 64, 64, 128)    │              0 │ batch_normalization_3… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ max_pooling2d_1           │ (None, 32, 32, 128)    │              0 │ activation_3[0][0]     │
│ (MaxPooling2D)            │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_4 (Conv2D)         │ (None, 32, 32, 256)    │        295,168 │ max_pooling2d_1[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_4

 Total params: 31,055,297 (118.47 MB)

 Trainable params: 31,043,521 (118.42 MB)

 Non-trainable params: 11,776 (46.00 KB)

In [ ]:
opt = tf.keras.optimizers.Adam(lr)
model.compile(loss="binary_crossentropy", optimizer=opt, metrics=["acc"])

In [ ]:
callbacks = [
    ModelCheckpoint(model_file, verbose=1, save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=4),
    CSVLogger(csv_file),
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=False)
]

In [ ]:
model.fit(
    train_dataset,
    epochs=epochs,
    validation_data=valid_dataset,
    callbacks=callbacks
)

Epoch 1/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - acc: 0.6824 - loss: 0.6569
Epoch 1: val_loss improved from inf to 0.64718, saving model to /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Files/Unet_model.keras
28/28 ━━━━━━━━━━━━━━━━━━━━ 124s 3s/step - acc: 0.6851 - loss: 0.6540 - val_acc: 0.7368 - val_loss: 0.6472 - learning_rate: 1.0000e-04
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 280ms/step - acc: 0.8462 - loss: 0.4064
Epoch 2: val_loss did not improve from 0.64718
28/28 ━━━━━━━━━━━━━━━━━━━━ 44s 328ms/step - acc: 0.8461 - loss: 0.4062 - val_acc: 0.6557 - val_loss: 0.6979 - learning_rate: 1.0000e-04
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 318ms/step - acc: 0.8608 - loss: 0.3256
Epoch 3: val_loss improved from 0.64718 to 0.53359, saving model to /content/drive/My Drive/Conductor_Image_Detection/Images/Lab_experiment/U-Net/Files/Unet_model.keras
28/28 ━━━━━━━━━━━━━━━━━━━━ 14s 493ms/step - acc: 0.8610 - loss: 0.3250 - val_acc: 0.7631 - val_loss: 0.533